# 07 — Results, plots, errors and runtime analysis

**Purpose:** Create report-ready tables and charts using only completed and validated training and inference records.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/Urdu_Qwen_Research_Notebooks'), Path('/content/Urdu_Qwen_Research_Notebooks')]
ROOT = next((p for p in CANDIDATES if (p / 'project_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Unzip the entire project folder into /kaggle/working/ or /content/, then run this notebook. See README.md.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print('Project folder:', ROOT)

This notebook does not invent missing metrics or training times. Plots appear only when result files exist. **Do not mix validation and test metrics**, or compare different task label sets as if they were the same experiment.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import Counter
RUNS=read_jsonl(ROOT/'outputs/metrics/train_runs.jsonl')
runs=pd.DataFrame([x for x in RUNS if not x.get('pilot') and x.get('status')=='complete'])
if len(runs):
    display(runs[['run_id','method','examples','lora_rank','seed','trainable_pct','wall_seconds','peak_gpu_allocated_gib']])
else: print('No completed training results yet.')

In [ ]:
paths=sorted((ROOT/'outputs/metrics').glob('*_leaderboard.csv'))
if paths:
    scores=pd.concat([pd.read_csv(x) for x in paths],ignore_index=True)
    display(scores[['run_id','lang','split','accuracy','macro_f1','invalid_count']])
    for key,group in scores.groupby(['task','lang','split']):
        ax=group.plot(kind='bar',x='run_id',y='macro_f1',legend=False,figsize=(max(7,len(group)*0.8),4))
        ax.set_title('Macro-F1 | '+str(key));ax.set_ylim(0,1);ax.set_ylabel('Macro-F1');
        plt.xticks(rotation=35,ha='right');plt.tight_layout()
        plt.savefig(ROOT/'outputs/plots'/('_'.join(key)+'_macro_f1.png'),dpi=160); plt.show()
else: print('No evaluated metrics yet. Run 05 then 06.')

## Paired prediction comparison

For a scientific comparison, models must be evaluated on the same test examples. This cell prints correct/incorrect disagreements for a pair. It does not claim significance from one seed or a tiny test set.

In [ ]:
COMPARE_A=None # example: 'sib200_lora_ur_n10_s42_r8_ur_test.csv'
COMPARE_B=None # example: 'sib200_full_ur_n10_s42_ur_test.csv'
if COMPARE_A and COMPARE_B:
    pa=ROOT/'outputs/predictions'/COMPARE_A; pb=ROOT/'outputs/predictions'/COMPARE_B
    ma,mb=read_json(pa.with_suffix('.meta.json')),read_json(pb.with_suffix('.meta.json'))
    if (ma['task'],ma['lang'],ma['split'],ma['dataset_sha256']) != (mb['task'],mb['lang'],mb['split'],mb['dataset_sha256']):
        raise ValueError('Different tasks/languages/splits/dataset hashes')
    da=pd.DataFrame(read_csv(pa)).set_index('example_id');db=pd.DataFrame(read_csv(pb)).set_index('example_id')
    assert set(da.index)==set(db.index),'Incomplete example alignment'
    aligned=da.join(db,lsuffix='_A',rsuffix='_B')
    a=(aligned['prediction_A']==aligned['reference_A'])
    b=(aligned['prediction_B']==aligned['reference_B'])
    print('Only A correct:',int((a & ~b).sum()), 'Only B correct:',int((~a & b).sum()))
    diff=aligned[a!=b]
    display(diff.head(30))
else: print('Set COMPARE_A/COMPARE_B to two complete prediction files.')

## Examine incorrectly classified Urdu examples

Review 20–50 errors manually for topic ambiguity, input length, misunderstood words, and invalid labels.

In [ ]:
ERROR_PREDICTION_FILE=None
if ERROR_PREDICTION_FILE:
    p=ROOT/'outputs/predictions'/ERROR_PREDICTION_FILE
    meta=read_json(p.with_suffix('.meta.json'))
    validate_prediction_metadata(meta,meta['task'],meta['lang'],meta['split'])
    reference=pd.DataFrame(load_task(meta['task'],meta['lang'],meta['split']))
    predicted=pd.DataFrame(read_csv(p))
    merged=predicted.merge(reference[['example_id','text','label']],on='example_id',validate='one_to_one')
    errors=merged[merged.prediction!=merged.label]
    errors[['example_id','text','label','prediction','raw_prediction']].head(50).to_csv(ROOT/'outputs/metrics'/'error_analysis_sample.csv',index=False)
    display(errors[['text','label','prediction','raw_prediction']].head(30))
else: print('Choose a validated inference file.')

## Actual GPU duration and memory (not predicted numbers)

In [ ]:
if len(runs):
    f=runs.copy(); f['minutes']=f.wall_seconds/60
    ax=f.plot(kind='bar',x='run_id',y='minutes',legend=False,figsize=(max(7,len(f)*0.8),4))
    ax.set_ylabel('Actual completed training minutes');plt.xticks(rotation=35,ha='right');plt.tight_layout()
    plt.savefig(ROOT/'outputs/plots'/'actual_train_times.png',dpi=160);plt.show()
else: print('Training time graphs require completed runs.')